# Appendix C: The Weil Conjectures

**Source span.** Printed pages 449-458; PDF pages 464-473. The source is used only for structure, terminology, theorem orientation, and page spans. All prose, code, examples, and generated artifacts are original.

## Appendix Goal

This appendix connects arithmetic point counts over finite fields to cohomological structure. The source introduces zeta functions, states the Weil conjectures, sketches the historical path through Dwork, Grothendieck, and Deligne, recalls the role of l-adic cohomology, and explains how Frobenius plus a Lefschetz trace formula turns point counts into determinants on cohomology. The notebook makes that chain executable in small examples.

## Translation Guide

A variety over `F_q` is translated into a sequence of counts `N_r = #X(F_{q^r})`. The zeta function is translated into a generating function whose logarithm stores those counts. Frobenius is translated into a map whose fixed points over the algebraic closure recover the same `N_r`. Cohomology is translated into vector spaces carrying Frobenius actions, and the trace formula turns fixed-point counts into alternating traces. The analogue of the Riemann hypothesis is translated into a statement about the absolute values of reciprocal roots.

## Source Coverage Ledger

The source span was re-read before this revision. The notebook covers these section anchors: The Zeta Function and the Weil Conjectures; History of Work on the Weil Conjectures; The l-adic Cohomology; Cohomological Interpretation of the Weil Conjectures. It uses original examples and generated visuals only.

## Library Routing

NumPy performs finite field and eigenvalue calculations. Matplotlib draws durable count, zeta, eigenvalue, and trace-formula visuals. NetworkX represents the cohomological dependency graph. Plotly supplies an interactive Frobenius-root lab because the learner should see how changing the trace affects the Hasse circle. Artifacts are saved under `artifacts/appendix-c/`.

## Visual Storyboard

1. Finite-field point-count table for projective line and sample elliptic curves.
2. Zeta coefficient dashboard showing how `N_r` populates the exponential series.
3. Frobenius eigenvalue circle for an elliptic curve over a finite field.
4. Cohomological trace formula dependency graph.
5. Interactive root lab for traces satisfying the Hasse bound.


In [ ]:
from pathlib import Path
import sys, json, math, cmath
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import plotly.graph_objects as go

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "utils" / "artifacts.py").exists():
        BOOK_ROOT = candidate
        break
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from utils.artifacts import assert_artifacts, display_artifact, image_stats, save_json, save_matplotlib, save_plotly_html, save_table
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "appendix-c"
for child in ["figures", "html", "checks", "tables"]:
    (ARTIFACT_ROOT / child).mkdir(parents=True, exist_ok=True)

def relpath(path): return Path(path).resolve().relative_to(BOOK_ROOT).as_posix()

def count_elliptic(q, A=1, B=1):
    count = 1  # point at infinity
    squares = {y*y % q for y in range(q)}
    for x in range(q):
        rhs = (x**3 + A*x + B) % q
        if rhs == 0:
            count += 1
        elif rhs in squares:
            count += 2
    return count

source_sections = [
    {"number": "1", "title": "The Zeta Function and the Weil Conjectures", "printed_start": 449, "pdf_start": 464},
    {"number": "2", "title": "History of Work on the Weil Conjectures", "printed_start": 451, "pdf_start": 466},
    {"number": "3", "title": "The l-adic Cohomology", "printed_start": 453, "pdf_start": 468},
    {"number": "4", "title": "Cohomological Interpretation of the Weil Conjectures", "printed_start": 454, "pdf_start": 469},
]
visual_storyboard = [
    {"concept": "finite-field point counts", "artifact": "figures/finite-field-point-count-dashboard.png", "invariant": "P1(F_q) has q+1 points"},
    {"concept": "zeta function coefficients", "artifact": "figures/zeta-coefficient-dashboard.png", "invariant": "log Z records N_r/r"},
    {"concept": "Frobenius eigenvalues", "artifact": "figures/frobenius-eigenvalue-circle.png", "invariant": "elliptic reciprocal roots have magnitude sqrt(q) in the checked sample"},
    {"concept": "cohomological trace formula", "artifact": "figures/cohomological-trace-formula-graph.png", "invariant": "fixed-point counts equal alternating cohomological traces in the toy ledger"},
    {"concept": "Hasse root lab", "artifact": "html/frobenius-root-hasse-lab.html", "invariant": "traces inside the Hasse interval plot on the sqrt(q) circle"},
]
source_coverage_path = save_json({"appendix": "Appendix C: The Weil Conjectures", "printed_span": "449-458", "pdf_span": "464-473", "sections": source_sections, "copyright_note": "Original notebook prose and generated artifacts; source used only for orientation."}, ARTIFACT_ROOT, "checks", "source-coverage.json")
storyboard_path = save_json({"visual_sequence": visual_storyboard}, ARTIFACT_ROOT, "checks", "visual-storyboard.json")
generated_artifacts = [source_coverage_path, storyboard_path]
ARTIFACT_ROOT


In [ ]:
qs = [3, 5, 7, 11, 13]
point_rows = []
for q in qs:
    n_p1 = q + 1
    n_e = count_elliptic(q)
    trace = q + 1 - n_e
    point_rows.append({"q": q, "P1_points": n_p1, "elliptic_points": n_e, "trace_a": trace, "hasse_bound": 2*math.sqrt(q)})
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
axes[0].plot(qs, [row["P1_points"] for row in point_rows], marker="o", label="P1: q+1", color="#1971c2")
axes[0].plot(qs, [row["elliptic_points"] for row in point_rows], marker="s", label="E: y^2=x^3+x+1", color="#f08c00")
axes[0].set_xlabel("q"); axes[0].set_ylabel("F_q points")
axes[0].set_title("Finite-field point counts")
axes[0].legend(); axes[0].grid(alpha=0.2)
axes[1].bar([str(q) for q in qs], [row["trace_a"] for row in point_rows], color="#6741d9")
axes[1].plot([str(q) for q in qs], [row["hasse_bound"] for row in point_rows], color="#9d0208", ls="--", label="2 sqrt(q)")
axes[1].plot([str(q) for q in qs], [-row["hasse_bound"] for row in point_rows], color="#9d0208", ls="--")
axes[1].set_title("Elliptic trace inside Hasse bounds")
axes[1].set_xlabel("q"); axes[1].set_ylabel("a = q+1-N1")
axes[1].legend(); axes[1].grid(axis="y", alpha=0.2)
count_path = save_matplotlib(fig, ARTIFACT_ROOT, "figures", "finite-field-point-count-dashboard.png")
plt.close(fig)
generated_artifacts.append(count_path)
display_artifact(count_path, width=780)


The first dashboard puts the appendix's arithmetic input on screen. For `P^1`, the count is exactly `q+1`. For the sample elliptic curve, the count differs from `q+1` by a trace `a`, and the displayed dashed lines show the Hasse bound. This is the point-count face of the Weil conjectures: arithmetic data is organized into a sequence that later becomes a zeta function and then a cohomological determinant.


In [ ]:
q = 5
N_p1 = {r: q**r + 1 for r in range(1, 7)}
N_ell_1 = count_elliptic(q)
a_trace = q + 1 - N_ell_1
roots = np.roots([1, -a_trace, q])
N_ell = {r: int(round(q**r + 1 - (roots[0]**r + roots[1]**r).real)) for r in range(1, 7)}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
rvals = np.array(list(N_p1))
axes[0].bar(rvals - .18, [N_p1[r] for r in rvals], width=.36, label="P1", color="#1971c2")
axes[0].bar(rvals + .18, [N_ell[r] for r in rvals], width=.36, label="elliptic model", color="#f08c00")
axes[0].set_title("Counts N_r over F_{q^r}, q=5")
axes[0].set_xlabel("r"); axes[0].set_ylabel("N_r")
axes[0].legend(); axes[0].grid(axis="y", alpha=0.2)
coeffs = [N_p1[r]/r for r in rvals]
axes[1].plot(rvals, coeffs, marker="o", color="#2b8a3e")
axes[1].set_title("Log zeta coefficients for P1: N_r/r")
axes[1].set_xlabel("r"); axes[1].set_ylabel("coefficient")
axes[1].grid(alpha=0.2)
zeta_path = save_matplotlib(fig, ARTIFACT_ROOT, "figures", "zeta-coefficient-dashboard.png")
plt.close(fig)
generated_artifacts.append(zeta_path)
display_artifact(zeta_path, width=780)


The zeta function stores all the `N_r` at once through the exponential generating expression. The second dashboard shows the coefficient `N_r/r` for `P^1`, where the rational form is already known, and compares it with the elliptic model's point-count sequence. The important inspection target is that the zeta function is not a new arbitrary invariant; it is a compact encoding of fixed-point counts of Frobenius powers.


In [ ]:
radius = math.sqrt(q)
theta = np.linspace(0, 2*np.pi, 400)
fig, ax = plt.subplots(figsize=(6.7, 6.7))
ax.plot(radius*np.cos(theta), radius*np.sin(theta), color="#adb5bd", lw=2, label="|alpha|=sqrt(q)")
ax.scatter([root.real for root in roots], [root.imag for root in roots], s=130, color="#9d0208", label="Frobenius eigenvalues")
for idx, root in enumerate(roots, 1):
    ax.annotate(f"alpha{idx}", (root.real, root.imag), xytext=(8,8), textcoords="offset points")
ax.axhline(0, color="#333", lw=.8); ax.axvline(0, color="#333", lw=.8)
ax.set_aspect("equal", adjustable="box")
ax.set_title(f"Frobenius eigenvalue circle for q={q}, trace a={a_trace}")
ax.legend(loc="upper right")
ax.grid(alpha=0.2)
eigen_path = save_matplotlib(fig, ARTIFACT_ROOT, "figures", "frobenius-eigenvalue-circle.png")
plt.close(fig)
generated_artifacts.append(eigen_path)
display_artifact(eigen_path, width=640)


For an elliptic curve, the zeta numerator is governed by two Frobenius eigenvalues whose sum is the trace and whose product is `q`. In the displayed sample, both lie on the circle of radius `sqrt(q)`. That is the curve case of the Weil Riemann hypothesis in a form the notebook can check numerically. In higher dimension, the weights change by cohomological degree, but the guiding idea remains: zeta roots should be explained by Frobenius acting on cohomology.


In [ ]:
G = nx.DiGraph()
G.add_edges_from([
    ("X over F_q", "geometric Frobenius"),
    ("geometric Frobenius", "fixed points of F^r"),
    ("fixed points of F^r", "N_r"),
    ("l-adic cohomology", "Frobenius action on H^i"),
    ("Frobenius action on H^i", "alternating traces"),
    ("alternating traces", "N_r"),
    ("alternating traces", "determinants P_i(t)"),
    ("determinants P_i(t)", "zeta function"),
])
pos = {"X over F_q": (0,2), "geometric Frobenius": (2,2), "fixed points of F^r": (4,2), "N_r": (6,2), "l-adic cohomology": (0,.5), "Frobenius action on H^i": (2.5,.5), "alternating traces": (5,.5), "determinants P_i(t)": (7.4,.5), "zeta function": (9.4,.5)}
fig, ax = plt.subplots(figsize=(12, 4.6))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=14, edge_color="#495057", width=1.5)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color="#1971c2", node_size=1800)
nx.draw_networkx_labels(G, pos, ax=ax, font_color="white", font_size=8)
ax.set_title("Cohomological interpretation of the Weil conjectures")
ax.axis("off")
trace_path = save_matplotlib(fig, ARTIFACT_ROOT, "figures", "cohomological-trace-formula-graph.png")
plt.close(fig)
generated_artifacts.append(trace_path)
display_artifact(trace_path, width=780)

traces = list(range(-int(2*radius), int(2*radius)+1))
root_x = []
root_y = []
labels = []
for a in traces:
    if abs(a) <= 2*radius:
        rr = np.roots([1, -a, q])
        root_x.extend([rr[0].real, rr[1].real, None])
        root_y.extend([rr[0].imag, rr[1].imag, None])
        labels.extend([f"a={a}", f"a={a}", None])
fig_html = go.Figure()
fig_html.add_trace(go.Scatter(x=radius*np.cos(theta), y=radius*np.sin(theta), mode="lines", name="sqrt(q) circle"))
fig_html.add_trace(go.Scatter(x=root_x, y=root_y, mode="markers", text=labels, hovertemplate="%{text}<br>%{x:.3f}+%{y:.3f}i<extra></extra>", name="roots for integer traces"))
fig_html.update_layout(title="Frobenius root Hasse lab for q=5", xaxis_title="real", yaxis_title="imaginary", yaxis={"scaleanchor":"x", "scaleratio":1}, template="plotly_white", height=560)
lab_path = save_plotly_html(fig_html, ARTIFACT_ROOT, "html", "frobenius-root-hasse-lab.html")
generated_artifacts.append(lab_path)
display_artifact(lab_path, width="100%", height=430)


The trace-formula graph shows the conceptual route from arithmetic to cohomology. Fixed points of Frobenius powers give `N_r`; l-adic cohomology gives vector spaces on which Frobenius acts; alternating traces recover the same counts; determinants of those actions give the rational factors of the zeta function. The Plotly lab shows the elliptic curve version: integer traces satisfying the Hasse bound produce roots on the `sqrt(q)` circle.


In [ ]:
p1_checks = {q0: q0 + 1 for q0 in qs}
hasse_checks = {row["q"]: bool(abs(row["trace_a"]) <= row["hasse_bound"] + 1e-12) for row in point_rows}
root_magnitudes = [abs(root) for root in roots]
trace_formula_toy = {"H0_trace": 1, "H1_trace": float(roots[0] + roots[1]), "H2_trace": q}
trace_formula_toy["N1_from_traces"] = trace_formula_toy["H0_trace"] - trace_formula_toy["H1_trace"] + trace_formula_toy["H2_trace"]
source_table_path = save_table(source_sections, ARTIFACT_ROOT, "tables", "appendix-c-source-coverage.csv")
count_table_path = save_table(point_rows, ARTIFACT_ROOT, "tables", "finite-field-point-counts.csv")
checks_path = save_json({
    "P1_point_counts": p1_checks,
    "elliptic_hasse_checks": hasse_checks,
    "q": q,
    "elliptic_trace_a": int(a_trace),
    "frobenius_root_magnitudes": [float(v) for v in root_magnitudes],
    "trace_formula_toy": trace_formula_toy,
    "zeta_count_sequence_P1": N_p1,
    "zeta_count_sequence_elliptic": N_ell,
}, ARTIFACT_ROOT, "checks", "weil-conjectures-checks.json")
generated_artifacts.extend([source_table_path, count_table_path, checks_path])
assert all(value == q0 + 1 for q0, value in p1_checks.items())
assert all(hasse_checks.values())
assert all(abs(abs(root) - radius) < 1e-8 for root in roots)
assert abs(trace_formula_toy["N1_from_traces"] - N_ell_1) < 1e-8
assert_artifacts(generated_artifacts)
records = []
for artifact in generated_artifacts:
    record = {"path": relpath(artifact), "exists": artifact.exists(), "bytes": artifact.stat().st_size}
    if artifact.suffix.lower() == ".png": record.update(image_stats(artifact))
    records.append(record)
final_sanity = {
    "appendix": "Appendix C: The Weil Conjectures",
    "source_span": {"printed": "449-458", "pdf": "464-473"},
    "artifacts": records,
    "checks": {
        "P1_count_q_plus_1_checked": True,
        "elliptic_hasse_bound_checked": True,
        "frobenius_roots_on_sqrt_q_circle": True,
        "trace_formula_toy_recovers_N1": True,
        "source_sections_covered": len(source_sections),
    },
    "standalone_contract": "original prose, generated visuals, and local checks; no copied source text or figures",
}
final_path = save_json(final_sanity, ARTIFACT_ROOT, "checks", "final-sanity.json")
final_sanity["artifacts"].append({"path": relpath(final_path), "exists": final_path.exists(), "bytes": final_path.stat().st_size})
save_json(final_sanity, ARTIFACT_ROOT, "checks", "final-sanity.json")
final_sanity


## Standalone Study Notes

Appendix C is a compact story about why point counts should have topology behind them. The first section defines the zeta function from the counts over finite extensions. The conjectures then predict rationality, a functional equation, a Riemann-hypothesis-type bound on roots, and compatibility with Betti numbers. The history section matters because each part required a different tool: curves were accessible through classical geometry and Riemann-Roch, rationality and functional equations came from p-adic and cohomological approaches, and the root-size statement required Deligne's proof.

The l-adic cohomology section supplies the missing topological-looking vector spaces in characteristic `p`. The cohomological interpretation section is the key mechanism: Frobenius fixed points give the counts, Lefschetz expresses fixed points as alternating traces on cohomology, and determinants of Frobenius actions produce the zeta factors. In the notebook, `P^1` is the control case, the elliptic sample shows the Hasse bound and roots on the `sqrt(q)` circle, and the dependency graph shows how these computations scale conceptually to higher-dimensional smooth projective varieties.


## Coverage Checkpoints

A reader should be able to trace the full chain from equations over a finite field to cohomological determinants. First count points over `F_q` and its extensions. Then encode those counts in `Z(X,t)`. Next identify Frobenius powers whose fixed points produce the same counts. Then use l-adic cohomology and the Lefschetz formula to rewrite counts as alternating traces. Finally read rational factors, functional equations, Betti numbers, and root-size constraints from the Frobenius action. The notebook's small examples verify each link in that chain.
These checkpoints are intentionally practical. When the definitions become abstract, return to the artifact table, read the invariant attached to each file, and ask which source-section claim would fail if that invariant were false. That habit is the notebook course's substitute for passive summary.

The most common mistake in this appendix is to view the Weil conjectures as isolated formulas. They are a coordinated prediction that point counts should behave as if a good cohomology theory supplied traces, determinants, duality, and weights. The examples keep that coordination visible.

This final check is small, but it fixes the order of ideas before the reader meets the general theory.


## Takeaways

Appendix C is a bridge from finite arithmetic to cohomological structure. The raw data are point counts over finite extensions. The zeta function packages those counts. Frobenius supplies the fixed-point mechanism. l-adic cohomology supplies vector spaces on which Frobenius has traces and determinants. The Weil conjectures predict rationality, a functional equation, Betti-number compatibility, and root sizes of the correct weight. In this notebook, `P^1` verifies the simplest count, the elliptic sample shows Hasse-bound behavior and Frobenius roots, and the trace-formula graph explains why cohomology is the natural organizing language.
